In [36]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (OneHotEncoder,StandardScaler)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from google.colab import drive
import warnings

warnings.filterwarnings('ignore')
print('Libraries imported successfully')

Libraries imported successfully


In [37]:
# Google drive connection

drive.mount('/content/drive')
file_path = "/content/drive/MyDrive/Credit_Risk_Project/data/raw/Loan_default.csv"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [38]:
# Dataset loading

try:
  df = pd.read_csv(file_path, encoding='latin-1')
  print('Dataset loaded succesfully')
except FileNotFoundError:
  print(f"Error: The file was not found at '{file_path}'")
except Exception as e:
  print(f"Error loading the dataset: {e}")

Dataset loaded succesfully


In [39]:
# Exploring dataset

print(f"\nDataset dimensions: {df.shape[0]} rows and {df.shape[1]} columns")
print("\nDataset first rows:")
print(df.head())


Dataset dimensions: 255347 rows and 18 columns

Dataset first rows:
       LoanID  Age  Income  LoanAmount  CreditScore  MonthsEmployed  \
0  I38PQUQS96   56   85994       50587          520              80   
1  HPSK72WA7R   69   50432      124440          458              15   
2  C1OZ6DPJ8Y   46   84208      129188          451              26   
3  V2KKSFM3UN   32   31713       44799          743               0   
4  EY08JDHTZP   60   20437        9139          633               8   

   NumCreditLines  InterestRate  LoanTerm  DTIRatio    Education  \
0               4         15.23        36      0.44   Bachelor's   
1               1          4.81        60      0.68     Master's   
2               3         21.17        24      0.31     Master's   
3               3          7.07        24      0.23  High School   
4               4          6.51        48      0.73   Bachelor's   

  EmploymentType MaritalStatus HasMortgage HasDependents LoanPurpose  \
0      Full-time      D

In [40]:
print(f"\nDataset info:")
print(df.info())


Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 255347 entries, 0 to 255346
Data columns (total 18 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   LoanID          255347 non-null  object 
 1   Age             255347 non-null  int64  
 2   Income          255347 non-null  int64  
 3   LoanAmount      255347 non-null  int64  
 4   CreditScore     255347 non-null  int64  
 5   MonthsEmployed  255347 non-null  int64  
 6   NumCreditLines  255347 non-null  int64  
 7   InterestRate    255347 non-null  float64
 8   LoanTerm        255347 non-null  int64  
 9   DTIRatio        255347 non-null  float64
 10  Education       255347 non-null  object 
 11  EmploymentType  255347 non-null  object 
 12  MaritalStatus   255347 non-null  object 
 13  HasMortgage     255347 non-null  object 
 14  HasDependents   255347 non-null  object 
 15  LoanPurpose     255347 non-null  object 
 16  HasCoSigner     255347 non-null  object 


In [41]:
# Missing values analysis

print(f"\nPercentage of missing values:")
missing_values = df.isnull().mean() * 100
print(missing_values[missing_values > 0 ] if missing_values.any() else "There are no missing values")

missing_report = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage": (df.isnull().sum()/len(df))*100
})

missing_report.sort_values(
    by="Percentage",
    ascending=False
)


Percentage of missing values:
There are no missing values


,Missing Values,Percentage
LoanID,0,0.0
Age,0,0.0
Income,0,0.0
LoanAmount,0,0.0
CreditScore,0,0.0
MonthsEmployed,0,0.0
NumCreditLines,0,0.0
InterestRate,0,0.0
LoanTerm,0,0.0
DTIRatio,0,0.0


In [42]:
# Separate the target variable
target = "Default"

X = df.drop(columns=target)
y = df[target]

In [43]:
print(X.shape)
print(y.shape)

(255347, 17)
(255347,)


In [44]:
# Remove ID's
X = X.drop(columns="LoanID")
X.columns

Index(['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed',
       'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio', 'Education',
       'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents',
       'LoanPurpose', 'HasCoSigner'],
      dtype='object')

In [45]:
numeric_features = [
    "Age",
    "Income",
    "LoanAmount",
    "CreditScore",
    "MonthsEmployed",
    "NumCreditLines",
    "InterestRate",
    "LoanTerm",
    "DTIRatio"
]

binary_features = [
    "HasMortgage",
    "HasDependents",
    "HasCoSigner"
]

categorical_features = [
    "Education",
    "EmploymentType",
    "MaritalStatus",
    "LoanPurpose"
]

target = "Default"

In [46]:
# We make sure we haven't left out any columns.
print("Numeric:", len(numeric_features))
print("Binary:", len(binary_features))
print("Categorical:", len(categorical_features))

Numeric: 9
Binary: 3
Categorical: 4


In [47]:
print(X.shape)

(255347, 16)


In [48]:
# Verify that our lists exactly match the columns in the DataFrame.
expected_features = (
    numeric_features
    + binary_features
    + categorical_features
)

print(set(X.columns) == set(expected_features))

True


In [49]:
# Conversion of binary variables
binary_mapping = {
    "Yes": 1,
    "No": 0
}

X[binary_features] = X[binary_features].replace(binary_mapping)

In [50]:
# Verifying binary conversion
X[binary_features].head()

,HasMortgage,HasDependents,HasCoSigner
0,1,1,1
1,0,0,1
2,1,1,0
3,0,0,0
4,0,1,0


In [51]:
# Verifying binary conversion with for loop
for column in binary_features:
    print(column)
    print(X[column].unique())
    print()

HasMortgage
[1 0]

HasDependents
[1 0]

HasCoSigner
[1 0]



In [52]:
# Building the preprocessor
preprocessor = ColumnTransformer(

    transformers=[

        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),

        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),

        (
            "binary",
            "passthrough",
            binary_features
        )

    ]

)

In [53]:
# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [54]:
# Verifying what we got from Train/Test split
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print()

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (204277, 16)
X_test : (51070, 16)

y_train: (204277,)
y_test : (51070,)


In [55]:
print(y_train.value_counts())
print()

print(y_test.value_counts())

Default
0    180555
1     23722
Name: count, dtype: int64

Default
0    45139
1     5931
Name: count, dtype: int64


In [56]:
# Verifying the same distribution in the sets (stratify).

print("Original distribution")
print(y.value_counts(normalize=True))

print()

print("Train distribution")
print(y_train.value_counts(normalize=True))

print()

print("Test distribution")
print(y_test.value_counts(normalize=True))

Original distribution
Default
0    0.883872
1    0.116128
Name: proportion, dtype: float64

Train distribution
Default
0    0.883873
1    0.116127
Name: proportion, dtype: float64

Test distribution
Default
0    0.883865
1    0.116135
Name: proportion, dtype: float64


In [57]:
# Applying the preprocessor
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [58]:
type(X_train_processed)
type(X_test_processed)

numpy.ndarray

In [59]:
print(X_train_processed.shape)
print(X_test_processed.shape)

(204277, 28)
(51070, 28)


In [60]:
feature_names = preprocessor.get_feature_names_out()

len(feature_names)

28

In [61]:
for feature in feature_names:
    print(feature)

numeric__Age
numeric__Income
numeric__LoanAmount
numeric__CreditScore
numeric__MonthsEmployed
numeric__NumCreditLines
numeric__InterestRate
numeric__LoanTerm
numeric__DTIRatio
categorical__Education_Bachelor's
categorical__Education_High School
categorical__Education_Master's
categorical__Education_PhD
categorical__EmploymentType_Full-time
categorical__EmploymentType_Part-time
categorical__EmploymentType_Self-employed
categorical__EmploymentType_Unemployed
categorical__MaritalStatus_Divorced
categorical__MaritalStatus_Married
categorical__MaritalStatus_Single
categorical__LoanPurpose_Auto
categorical__LoanPurpose_Business
categorical__LoanPurpose_Education
categorical__LoanPurpose_Home
categorical__LoanPurpose_Other
binary__HasMortgage
binary__HasDependents
binary__HasCoSigner


In [62]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_train_processed_df.head()

,numeric__Age,numeric__Income,numeric__LoanAmount,numeric__CreditScore,numeric__MonthsEmployed,numeric__NumCreditLines,numeric__InterestRate,numeric__LoanTerm,numeric__DTIRatio,categorical__Education_Bachelor's,...,categorical__MaritalStatus_Married,categorical__MaritalStatus_Single,categorical__LoanPurpose_Auto,categorical__LoanPurpose_Business,categorical__LoanPurpose_Education,categorical__LoanPurpose_Home,categorical__LoanPurpose_Other,binary__HasMortgage,binary__HasDependents,binary__HasCoSigner
15826,0.099486,-1.167307,1.698627,0.312274,-1.170591,-0.449335,-1.335928,1.414459,1.512942,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
147371,0.299620,1.319747,-0.864170,-0.505800,1.714171,0.445947,0.185568,0.707489,-0.045473,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0
178180,0.232908,0.453497,-1.700954,0.903803,1.396847,0.445947,-1.201856,-0.706451,1.123338,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0
126915,-0.100649,-1.191966,-1.432895,-1.449730,-1.661000,0.445947,0.723364,0.000519,1.123338,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
163930,-1.568299,0.434508,1.707671,-1.613345,0.416028,0.445947,0.898110,1.414459,-0.218630,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [63]:
# Remove LoanTerm because it does not contribute to the analysis
X = X.drop(columns=["LoanTerm"])

In [64]:
# Upgrade lists
numeric_features = [
    "Age",
    "Income",
    "LoanAmount",
    "CreditScore",
    "MonthsEmployed",
    "NumCreditLines",
    "InterestRate",
    "DTIRatio"
]

binary_features = [
    "HasMortgage",
    "HasDependents",
    "HasCoSigner"
]

categorical_features = [
    "Education",
    "EmploymentType",
    "MaritalStatus",
    "LoanPurpose"
]

In [65]:
# Rebuild the preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "binary",
            "passthrough",
            binary_features
        )
    ]
)

In [66]:
# Split again
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [67]:
# Applying preprocessor again
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [68]:
# Final check
print(X_train_processed.shape)

print(X_test_processed.shape)

(204277, 27)
(51070, 27)


In [70]:
# Saving preprocessor
import joblib

joblib.dump(preprocessor, "/content/drive/MyDrive/Credit_Risk_Project/models/preprocessor.pkl")


['/content/drive/MyDrive/Credit_Risk_Project/models/preprocessor.pkl']

In [72]:
# Saving sets processed
np.save("/content/drive/MyDrive/Credit_Risk_Project/data/processed/X_train_processed.npy", X_train_processed)
np.save("/content/drive/MyDrive/Credit_Risk_Project/data/processed/X_test_processed.npy", X_test_processed)

y_train.to_csv("/content/drive/MyDrive/Credit_Risk_Project/data/processed/y_train.csv", index=False)
y_test.to_csv("/content/drive/MyDrive/Credit_Risk_Project/data/processed/y_test.csv", index=False)

In [73]:
# Saving label names
feature_names = preprocessor.get_feature_names_out()

feature_names = pd.DataFrame(
    feature_names,
    columns=["Feature"]
)

feature_names.to_csv(
    "/content/drive/MyDrive/Credit_Risk_Project/data/processed/feature_names.csv",
    index=False
)